# Kaggle — SE-ResNet-18 + Log-Mel | ASVspoof 2019 LA

In [ ]:
import subprocess
subprocess.run(["pip", "install", "soundfile", "librosa", "-q"])
import os, numpy as np, pandas as pd, soundfile as sf, librosa
import scipy.fftpack as fft_
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import roc_curve, roc_auc_score
import time, json, math

# ── verified paths ──────────────────────────────────────────────────────────
LA_ROOT    = "/kaggle/input/asvpoof-2019-dataset/LA"
PROTO_DIR  = f"{LA_ROOT}/ASVspoof2019_LA_cm_protocols"
TRAIN_FLAC = f"{LA_ROOT}/ASVspoof2019_LA_train/flac"
DEV_FLAC   = f"{LA_ROOT}/ASVspoof2019_LA_dev/flac"
EVAL_FLAC  = f"{LA_ROOT}/ASVspoof2019_LA_eval/flac"

# verify
for p in [LA_ROOT, PROTO_DIR, TRAIN_FLAC, DEV_FLAC, EVAL_FLAC]:
    exists = os.path.exists(p)
    count  = len(os.listdir(p)) if exists else 0
    print(f"{'OK' if exists else 'MISSING'} ({count:6d} items)  {p}")


In [ ]:
# protocol filenames verified from Kaggle dataset page
PROTO_FILES = {
    "train": f"{PROTO_DIR}/ASVspoof2019.LA.cm.train.trn.txt",
    "dev"  : f"{PROTO_DIR}/ASVspoof2019.LA.cm.dev.trl.txt",
    "eval" : f"{PROTO_DIR}/ASVspoof2019.LA.cm.eval.trl.txt",
}
FLAC_DIRS = {
    "train": TRAIN_FLAC,
    "dev"  : DEV_FLAC,
    "eval" : EVAL_FLAC,
}

rows = []
for partition, proto_path in PROTO_FILES.items():
    flac_dir = FLAC_DIRS[partition]
    with open(proto_path, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            spk, aid, _, atk, key = parts[0], parts[1], parts[2], parts[3], parts[4]
            rows.append({
                "speaker_id": spk,
                "audio_id"  : aid,
                "attack_id" : atk,
                "key"       : key,
                "is_spoof"  : int(key == "spoof"),
                "partition" : partition,
                "file_path" : f"{flac_dir}/{aid}.flac",
            })

manifest = pd.DataFrame(rows)
print(f"Total rows: {len(manifest)}")
for part in ["train", "dev", "eval"]:
    sub = manifest[manifest["partition"] == part]
    bon = len(sub[sub["key"] == "bonafide"])
    spf = len(sub[sub["key"] == "spoof"])
    print(f"  {part:5s}: {len(sub):6d} total | bon={bon:5d} spoof={spf:6d} ratio={spf/bon:.1f}:1")


In [ ]:
def load_audio(path, sr=16000):
    y, orig_sr = sf.read(path)
    if y.ndim > 1:
        y = y.mean(axis=1)
    y = y.astype(np.float32)
    if orig_sr != sr:
        y = librosa.resample(y, orig_sr=orig_sr, target_sr=sr)
    return y

def process(y, is_train=False, alpha=0.97, target=64000, top_db=40):
    y = np.concatenate([[y[0]], y[1:] - alpha * y[:-1]])
    ivs = librosa.effects.split(y=y, top_db=top_db)
    if len(ivs):
        t = np.concatenate([y[s:e] for s, e in ivs])
        if len(t) > 1000:
            y = t
    n = len(y)
    if n >= target:
        st = np.random.randint(0, n - target + 1) if is_train else (n - target) // 2
        y = y[st:st + target]
    else:
        y = np.pad(y, (0, target - n), mode="wrap")
    return y / (np.max(np.abs(y)) + 1e-7)

def extract_lfcc(y, sr=16000, n_fft=1024, hop=256, n_lfcc=20, T=251):
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop, center=True)) ** 2
    nb = S.shape[0]
    fb = np.zeros((n_lfcc, nb), dtype=np.float32)
    pts = np.linspace(0, nb - 1, n_lfcc + 2, dtype=int)
    for i in range(n_lfcc):
        lo, mid, hi = pts[i], pts[i+1], pts[i+2]
        if mid > lo: fb[i, lo:mid] = np.linspace(0, 1, mid - lo)
        if hi > mid: fb[i, mid:hi] = np.linspace(1, 0, hi - mid)
    log_e  = np.log(np.maximum(fb @ S, 1e-8))
    static = fft_.dct(log_e, type=2, axis=0, norm="ortho")[:n_lfcc]
    d1     = librosa.feature.delta(static, order=1)
    d2     = librosa.feature.delta(static, order=2)
    feat   = np.vstack([static, d1, d2]).astype(np.float32)
    feat   = feat[:, :T] if feat.shape[1] >= T else np.pad(feat, ((0,0),(0,T-feat.shape[1])), mode="edge")
    return feat   # shape (60, 251)

def extract_mel(y, sr=16000, n_fft=1024, hop=256, n_mels=128, fmin=20, fmax=8000, T=251):
    M   = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft, hop_length=hop,
                                          n_mels=n_mels, fmin=fmin, fmax=fmax)
    lM  = librosa.power_to_db(M, ref=np.max).astype(np.float32)
    lM  = lM[:, :T] if lM.shape[1] >= T else np.pad(lM, ((0,0),(0,T-lM.shape[1])), mode="edge")
    return lM   # shape (128, 251)

# sanity check
row = manifest[manifest["partition"] == "train"].iloc[0]
_y  = process(load_audio(row["file_path"]))
_lf = extract_lfcc(_y)
_ml = extract_mel(_y)
print(f"LFCC: {_lf.shape}  (expected (60, 251))")
print(f"Mel : {_ml.shape}  (expected (128, 251))")
assert _lf.shape == (60,251) and _ml.shape == (128,251)
print("Audio pipeline OK.")
del _y, _lf, _ml


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, ls=0.05):
        super().__init__()
        self.a, self.g, self.ls = alpha, gamma, ls
    def forward(self, inp, tgt):
        ce = F.cross_entropy(inp, tgt, reduction="none", label_smoothing=self.ls)
        pt = torch.exp(-ce)
        at = torch.where(tgt == 1, self.a, 1.0 - self.a)
        return (at * (1 - pt) ** self.g * ce).mean()

def eer(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score, pos_label=1)
    fnr = 1 - tpr
    i   = np.nanargmin(np.abs(fpr - fnr))
    return float((fpr[i] + fnr[i]) / 2)

def weighted_sampler(labels):
    cc  = np.bincount(labels)
    sw  = torch.FloatTensor((1.0 / cc)[labels])
    return WeightedRandomSampler(sw, len(sw), replacement=True)

def eval_model(model, loader, device, use_amp):
    model.eval()
    probs, targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            with torch.cuda.amp.autocast(enabled=use_amp):
                p = torch.softmax(model(xb.to(device)), dim=1)[:, 1].cpu().numpy()
            probs.append(p); targets.append(yb.numpy())
    y_prob = np.concatenate(probs); y_true = np.concatenate(targets)
    return eer(y_true, y_prob), roc_auc_score(y_true, y_prob), y_prob, y_true

def train_model(model, train_loader, dev_loader, train_ds, criterion, optimizer,
                scheduler, scaler, cfg, use_amp, device):
    best_eer, best_auc = float("inf"), 0.0
    history = []
    save_path = f"/kaggle/working/{cfg['name']}_best.pth"
    os.makedirs("/kaggle/working", exist_ok=True)
    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        total_loss, t0 = 0.0, time.time()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=use_amp):
                loss = criterion(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            total_loss += loss.item() * xb.size(0)
        train_loss = total_loss / len(train_ds)
        scheduler.step(epoch)
        e, auc, _, _ = eval_model(model, dev_loader, device, use_amp)
        elapsed = time.time() - t0
        print(f"Ep {epoch:02d}/{cfg['epochs']} | loss={train_loss:.4f} | EER={e*100:.2f}% | AUC={auc:.4f} | {elapsed:.0f}s")
        history.append({"epoch": epoch, "train_loss": round(train_loss,6),
                        "val_eer": round(e,6), "val_auc": round(auc,6)})
        if e < best_eer:
            best_eer, best_auc = e, auc
            torch.save({"epoch": epoch, "state_dict": model.state_dict(),
                        "eer": best_eer, "auc": best_auc, "cfg": cfg}, save_path)
            print(f"  >>> Best: EER={best_eer*100:.2f}% AUC={best_auc:.4f}")
    json.dump(history, open(f"/kaggle/working/{cfg['name']}_history.json", "w"), indent=2)
    print(f"\nDone. Best EER={best_eer*100:.2f}% AUC={best_auc:.4f}  →  {save_path}")
    return history, best_eer, best_auc

def plot_history(history, name):
    import matplotlib.pyplot as plt
    ep = [h["epoch"] for h in history]
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    ax[0].plot(ep, [h["train_loss"] for h in history], "#3498db", lw=2)
    ax[0].set(title="Training Loss", xlabel="Epoch")
    eers = [h["val_eer"]*100 for h in history]
    ax[1].plot(ep, eers, "#e74c3c", lw=2, marker="o", ms=3)
    ax[1].axhline(min(eers), color="gray", ls="--", label=f"Best: {min(eers):.2f}%")
    ax[1].set(title="Dev EER (%)", xlabel="Epoch"); ax[1].legend()
    aucs = [h["val_auc"] for h in history]
    ax[2].plot(ep, aucs, "#2ecc71", lw=2, marker="o", ms=3)
    ax[2].axhline(max(aucs), color="gray", ls="--", label=f"Best: {max(aucs):.4f}")
    ax[2].set(title="Dev AUC", xlabel="Epoch"); ax[2].legend()
    plt.tight_layout()
    plt.savefig(f"/kaggle/working/{name}_curve.png", dpi=150)
    plt.show()


In [ ]:
class MelDataset(Dataset):
    def __init__(self, df, is_train=False):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            y = process(load_audio(row["file_path"]), self.is_train)
            x = extract_mel(y)
        except Exception:
            x = np.zeros((128, 251), dtype=np.float32)
        if self.is_train:
            if np.random.rand() < 0.5:
                t = np.random.randint(1, 31); t0 = np.random.randint(0, max(1, 251-t))
                x[:, t0:t0+t] = x.mean()
            if np.random.rand() < 0.5:
                f = np.random.randint(1, 16); f0 = np.random.randint(0, max(1, 128-f))
                x[f0:f0+f, :] = x.mean()
        return torch.from_numpy(x).unsqueeze(0), torch.tensor(int(row["is_spoof"]), dtype=torch.long)

train_df = manifest[manifest["partition"] == "train"].reset_index(drop=True)
dev_df   = manifest[manifest["partition"] == "dev"].reset_index(drop=True)
train_ds = MelDataset(train_df, is_train=True)
dev_ds   = MelDataset(dev_df,   is_train=False)
sampler  = weighted_sampler(train_df["is_spoof"].values)

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print(f"Device: {DEVICE}")

BATCH = 64
train_loader = DataLoader(train_ds, batch_size=BATCH,   sampler=sampler, num_workers=2, pin_memory=True)
dev_loader   = DataLoader(dev_ds,   batch_size=BATCH*2, shuffle=False,   num_workers=2, pin_memory=True)
print(f"Train batches: {len(train_loader)} | Dev batches: {len(dev_loader)}")


In [ ]:
class SEBlock(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        self.fc = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Linear(ch, ch//r), nn.ReLU(),
                                  nn.Linear(ch//r, ch), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x).view(x.size(0), x.size(1), 1, 1)

class ResBlk(nn.Module):
    def __init__(self, ic, oc, s=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(ic, oc, 3, s, 1, bias=False), nn.BatchNorm2d(oc), nn.ReLU(),
            nn.Conv2d(oc, oc, 3, 1, 1, bias=False), nn.BatchNorm2d(oc))
        self.se   = SEBlock(oc)
        self.skip = nn.Sequential(nn.Conv2d(ic, oc, 1, s, bias=False),
                                   nn.BatchNorm2d(oc)) if s != 1 or ic != oc else nn.Identity()
    def forward(self, x):
        return F.relu(self.se(self.conv(x)) + self.skip(x))

class SEResNet18(nn.Module):
    def __init__(self, drop=0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, 32, 7, 2, 3, bias=False),
                                   nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(3, 2, 1))
        self.l1 = nn.Sequential(ResBlk(32,  32),        ResBlk(32,  32))
        self.l2 = nn.Sequential(ResBlk(32,  64,  s=2),  ResBlk(64,  64))
        self.l3 = nn.Sequential(ResBlk(64,  128, s=2),  ResBlk(128, 128))
        self.l4 = nn.Sequential(ResBlk(128, 256, s=2),  ResBlk(256, 256))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                   nn.Dropout(drop), nn.Linear(256, 128), nn.ReLU(),
                                   nn.Linear(128, 2))
    def forward(self, x):
        x = self.stem(x)
        for l in [self.l1, self.l2, self.l3, self.l4]:
            x = l(x)
        return self.head(x)

model = SEResNet18().to(DEVICE if "DEVICE" in dir() else torch.device("cpu"))
n = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"SE-ResNet-18 params: {n:,}")
with torch.no_grad():
    dummy = torch.zeros(2, 1, 128, 251).to(next(model.parameters()).device)
    print(f"Forward pass: {model(dummy).shape}  (expected [2, 2])")


In [ ]:
CFG = {"name": "se_resnet_mel", "epochs": 30, "lr": 5e-4, "lr_min": 1e-6, "wd": 1e-4}
criterion = FocalLoss(0.75, 2.0, 0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["wd"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=CFG["lr_min"])
scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP)

history, best_eer, best_auc = train_model(
    model, train_loader, dev_loader, train_ds,
    criterion, optimizer, scheduler, scaler, CFG, USE_AMP, DEVICE)


In [ ]:
plot_history(history, 'se_resnet_mel')